# P3 · Drug–Gene Interaction Collection
**Input:** `results/tables/hub_genes.csv` · `results/tables/survival_filtered_genes.csv`  
**Outputs:** `results/tables/dgi_edges_gnn.csv` · `results/figures/dgi_summary_dashboard.png`

**Run order:** P1 → P2 → **P3** → P4


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paths import REPO_ROOT, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
sys.path.insert(0, str(REPO_ROOT / "scripts"))
print(f"Repo root : {REPO_ROOT}")

In [ ]:
from dgi_functions import (load_dgi_inputs, collect_interactions,
                           build_dgi_dataframe, build_gnn_edge_list,
                           plot_dgi_dashboard)
print("Imports OK")

In [ ]:
# ── Database selection ────────────────────────────────────────────────────────
USE_DGIDB       = True   # DGIdb GraphQL API
USE_CHEMBL      = True   # ChEMBL REST API
USE_OPENTARGETS = True   # OpenTargets GraphQL API
USE_CURATED     = True   # Built-in curated fallback (fills gaps automatically)

# ── Composite score weights (must sum to 1.0) ─────────────────────────────────
W = {"interaction": 0.35, "publications": 0.20,
     "phase": 0.20, "approved": 0.15, "hub": 0.10}

print("Database selection:")
for name, flag in [("DGIdb", USE_DGIDB), ("ChEMBL", USE_CHEMBL),
                   ("OpenTargets", USE_OPENTARGETS), ("Curated", USE_CURATED)]:
    print(f"  {name:<14}: {'✓ enabled' if flag else '✗ disabled'}")

## 1 · Load gene lists

In [ ]:
gene_list, hub_score_map, surv_genes = load_dgi_inputs(TABLES_DIR)

## 2 · Query selected databases

In [ ]:
all_edges, apis_ok = collect_interactions(
    gene_list,
    use_dgidb=USE_DGIDB,
    use_chembl=USE_CHEMBL,
    use_opentargets=USE_OPENTARGETS,
    use_curated=USE_CURATED,
)

## 3 · Build & score edge dataframe

In [ ]:
dgi_df = build_dgi_dataframe(all_edges, hub_score_map, surv_genes, W)
dgi_df[["gene","drug","composite_score","approved","clinical_phase","source"]].head(10)

## 4 · Summary dashboard

In [ ]:
plot_dgi_dashboard(dgi_df, FIGURES_DIR)

## 5 · Export GNN-ready edge list

In [ ]:
gnn_df = build_gnn_edge_list(dgi_df, hub_score_map, surv_genes, TABLES_DIR)